# Experiment Leaderboard & Thesis Reporting

This notebook automatically discovers all experiment outputs from Google Drive,
builds a full leaderboard, generates ablation comparison tables and figures,
and produces a summary report for thesis writing.

**Project:** Explainable Multimodal Deep Learning for Vietnamese Restaurant Review Quality Regression

**Output:** 7 figures, 4 tables (CSV + XLSX), and a Markdown report saved to `reports/`.

---
## Section 1: Setup

Mount Google Drive, define paths, create output directories, and import libraries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q openpyxl pyyaml

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ---------- configurable paths ----------
DRIVE_ROOT = "/content/drive/MyDrive/SE365"
EXPERIMENTS_DIR = f"{DRIVE_ROOT}/experiments"
REPORTS_DIR = f"{DRIVE_ROOT}/reports"
FIGURES_DIR = f"{REPORTS_DIR}/figures"
TABLES_DIR = f"{REPORTS_DIR}/tables"

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

# ---------- plot style ----------
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.facecolor': 'white',
})

print(f"Experiments dir : {EXPERIMENTS_DIR}")
print(f"Reports dir     : {REPORTS_DIR}")

---
## Section 2: Experiment Discovery

Scan experiment folders, load `metrics.json`, optionally load `test_metrics.json` and `config.yaml`/`config.json`.

In [ ]:
def load_json(path):
    """Load a JSON file; return None on failure."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return None


def load_config(exp_dir):
    """Load config.yaml or config.json from an experiment directory."""
    yaml_path = os.path.join(exp_dir, 'config.yaml')
    json_path = os.path.join(exp_dir, 'config.json')
    if os.path.isfile(yaml_path):
        try:
            import yaml
            with open(yaml_path, 'r', encoding='utf-8') as f:
                return yaml.safe_load(f)
        except Exception:
            return None
    if os.path.isfile(json_path):
        return load_json(json_path)
    return None


def normalize_metrics(m):
    """Ensure standard metric keys exist, computing from available data."""
    if m is None:
        return None
    out = dict(m)

    # overall_mae / mae_overall
    if 'overall_mae' not in out and 'mae_overall' in out:
        out['overall_mae'] = out['mae_overall']
    elif 'mae_overall' not in out and 'overall_mae' in out:
        out['mae_overall'] = out['overall_mae']

    # aspect MAEs
    aspect_keys = ['mae_food', 'mae_price', 'mae_atmos', 'mae_service']
    available_aspects = [out[k] for k in aspect_keys if k in out]

    if 'aspect_mae' not in out and len(available_aspects) > 0:
        out['aspect_mae'] = float(np.mean(available_aspects))

    # mean_mae = average of all 5 target MAEs
    all_mae_keys = aspect_keys + ['mae_overall']
    available_all = [out[k] for k in all_mae_keys if k in out]
    if 'mean_mae' not in out and len(available_all) > 0:
        out['mean_mae'] = float(np.mean(available_all))

    return out


# ---------- discover ----------
experiment_records = []
skipped_folders = []

if not os.path.isdir(EXPERIMENTS_DIR):
    raise FileNotFoundError(f"Experiments directory not found: {EXPERIMENTS_DIR}")

folders = sorted([
    d for d in os.listdir(EXPERIMENTS_DIR)
    if os.path.isdir(os.path.join(EXPERIMENTS_DIR, d))
])

print(f"Found {len(folders)} experiment folder(s).\n")

for folder_name in folders:
    exp_path = os.path.join(EXPERIMENTS_DIR, folder_name)
    metrics_path = os.path.join(exp_path, 'metrics.json')

    if not os.path.isfile(metrics_path):
        skipped_folders.append(folder_name)
        continue

    raw_metrics = load_json(metrics_path)
    if raw_metrics is None:
        skipped_folders.append(folder_name)
        continue

    metrics = normalize_metrics(raw_metrics)

    # test metrics
    test_metrics_path = os.path.join(exp_path, 'test_metrics.json')
    test_metrics = None
    has_test = False
    if os.path.isfile(test_metrics_path):
        test_metrics = normalize_metrics(load_json(test_metrics_path))
        has_test = test_metrics is not None

    config = load_config(exp_path)

    record = {
        'experiment_id': folder_name,
        'config': config,
        'has_test_metrics': has_test,
    }
    # flatten val metrics
    for k, v in metrics.items():
        record[k] = v
    # flatten test metrics with prefix
    if test_metrics:
        for k, v in test_metrics.items():
            record[f'test_{k}'] = v

    experiment_records.append(record)

print(f"Loaded metrics for {len(experiment_records)} experiment(s).")
if skipped_folders:
    print(f"\nSkipped (no metrics.json): {skipped_folders}")

---
## Section 3: Metadata Parsing

Infer phase, image backbone, text backbone, fusion method, and loss function from experiment names and configs.

In [ ]:
# ---------- mapping tables ----------
IMAGE_BACKBONE_MAP = {
    'convnext': 'ConvNeXt',
    'swinb': 'Swin-B',
    'swin_base': 'Swin-B',
    'efficientnetb3': 'EfficientNet-B3',
    'efficientnet_b3': 'EfficientNet-B3',
    'siglip': 'SigLIP',
}

TEXT_BACKBONE_MAP = {
    'xlmr': 'XLM-R',
    'xlm-roberta': 'XLM-R',
    'xlm_roberta': 'XLM-R',
    'phobert': 'PhoBERT',
    'visobert': 'ViSoBERT',
}

FUSION_MAP = {
    'concat': 'Concat',
    'gmu': 'GMU',
    'gatedcrossmodal': 'Gated Cross-Modal',
    'gated_cross': 'Gated Cross-Modal',
    'film': 'FiLM',
    'crossattention': 'Cross-Attention',
    'cross_attention': 'Cross-Attention',
}

LOSS_MAP = {
    'mse': 'MSE',
    'huber': 'Huber',
    'logcosh': 'Log-Cosh',
    'log_cosh': 'Log-Cosh',
    'uncertaintyweighted': 'Uncertainty Weighted',
    'uncertainty': 'Uncertainty Weighted',
    'auto_weight': 'AutoWeight',
    'autoweight': 'AutoWeight',
}


def _match_from_map(text, mapping):
    """Find the first mapping key present in text (lowercased)."""
    text_lower = text.lower()
    # sort by key length descending so longer matches win
    for key in sorted(mapping.keys(), key=len, reverse=True):
        if key in text_lower:
            return mapping[key]
    return None


def infer_phase(exp_id):
    """Determine the experiment phase from its ID prefix."""
    eid = exp_id.upper()
    if 'EXP_010' in eid:
        if 'text_only' in exp_id.lower():
            return 'Baseline (Text-Only)'
        return 'Baseline'
    if 'EXP_011' in eid:
        if 'image_only' in exp_id.lower():
            return 'Baseline (Image-Only)'
        return 'Baseline'
    if 'EXP_012' in eid:
        return 'Baseline (Multimodal)'
    if 'EXP_020' in eid:
        return 'Image Ablation'
    if 'EXP_030' in eid:
        return 'Text Ablation'
    if 'EXP_040' in eid or 'EXP_041' in eid:
        return 'Fusion Ablation'
    if 'EXP_050' in eid or 'EXP_051' in eid:
        return 'Loss Ablation'
    if 'EXP_060' in eid:
        return 'Promising Combination'
    if 'EXP_070' in eid:
        return 'Seed Validation'
    return 'Unknown'


def infer_image_backbone(exp_id, config):
    # try config first
    if config:
        model_name = config.get('image_model_name', '')
        result = _match_from_map(model_name, IMAGE_BACKBONE_MAP)
        if result:
            return result
    # fallback to folder name
    result = _match_from_map(exp_id, IMAGE_BACKBONE_MAP)
    if result:
        return result
    # special: bestimage means inherited from phase-2 winner (Swin-B)
    if 'bestimage' in exp_id.lower():
        return 'Swin-B'
    # text-only has no image backbone
    if 'text_only' in exp_id.lower():
        return 'N/A'
    # if config has image_model_name, return it raw
    if config and config.get('image_model_name'):
        return config['image_model_name']
    return 'Unknown'


def infer_text_backbone(exp_id, config):
    if config:
        model_name = config.get('text_model_name', '')
        result = _match_from_map(model_name, TEXT_BACKBONE_MAP)
        if result:
            return result
    result = _match_from_map(exp_id, TEXT_BACKBONE_MAP)
    if result:
        return result
    if 'besttext' in exp_id.lower():
        return 'PhoBERT'
    if 'image_only' in exp_id.lower():
        return 'N/A'
    if config and config.get('text_model_name'):
        return config['text_model_name']
    return 'Unknown'


def infer_fusion(exp_id, config):
    if config:
        ft = config.get('fusion_type', '')
        result = _match_from_map(ft, FUSION_MAP)
        if result:
            return result
    result = _match_from_map(exp_id, FUSION_MAP)
    if result:
        return result
    # bestfusion = phase-4 winner (Cross-Attention)
    if 'bestfusion' in exp_id.lower() or 'bestsequential' in exp_id.lower():
        return 'Cross-Attention'
    # unimodal
    if 'text_only' in exp_id.lower() or 'image_only' in exp_id.lower():
        return 'N/A'
    return 'Unknown'


def infer_loss(exp_id, config):
    if config:
        lf = config.get('loss_fn', '')
        result = _match_from_map(lf, LOSS_MAP)
        if result:
            return result
    result = _match_from_map(exp_id, LOSS_MAP)
    if result:
        return result
    return 'Unknown'


def is_unimodal_text(exp_id):
    return 'text_only' in exp_id.lower()


def is_unimodal_image(exp_id):
    return 'image_only' in exp_id.lower()


# ---------- apply ----------
for rec in experiment_records:
    eid = rec['experiment_id']
    cfg = rec.get('config')
    rec['phase'] = infer_phase(eid)
    rec['image_backbone'] = infer_image_backbone(eid, cfg)
    rec['text_backbone'] = infer_text_backbone(eid, cfg)
    rec['fusion_method'] = infer_fusion(eid, cfg)
    rec['loss_function'] = infer_loss(eid, cfg)
    rec['is_unimodal_text'] = is_unimodal_text(eid)
    rec['is_unimodal_image'] = is_unimodal_image(eid)

# build dataframe
df = pd.DataFrame(experiment_records)

# drop the raw config column
if 'config' in df.columns:
    df = df.drop(columns=['config'])

print(f"DataFrame shape: {df.shape}")
df[['experiment_id', 'phase', 'image_backbone', 'text_backbone',
    'fusion_method', 'loss_function', 'mean_mae']].head(25)

---
## Section 4: Full Experiment Leaderboard

Build the master leaderboard sorted by `mean_mae` (ascending). Save as CSV and XLSX, and plot a horizontal bar chart of the top 15.

In [ ]:
LEADERBOARD_COLS = [
    'rank', 'experiment_id', 'phase',
    'image_backbone', 'text_backbone', 'fusion_method', 'loss_function',
    'mean_mae', 'overall_mae', 'aspect_mae',
    'mae_food', 'mae_price', 'mae_service', 'mae_atmos',
    'rmse_overall', 'r2_overall', 'has_test_metrics',
]

df_lb = df.sort_values('mean_mae', ascending=True).reset_index(drop=True)
df_lb['rank'] = df_lb.index + 1

# keep only columns that exist
existing_cols = [c for c in LEADERBOARD_COLS if c in df_lb.columns]
df_leaderboard = df_lb[existing_cols].copy()

# save
csv_path = os.path.join(TABLES_DIR, 'full_experiment_leaderboard.csv')
xlsx_path = os.path.join(TABLES_DIR, 'full_experiment_leaderboard.xlsx')
df_leaderboard.to_csv(csv_path, index=False)
df_leaderboard.to_excel(xlsx_path, index=False, engine='openpyxl')
print(f"Saved: {csv_path}")
print(f"Saved: {xlsx_path}")

print("\n=== Top 10 Experiments ===")
df_leaderboard.head(10)

In [ ]:
# ---------- Figure 1: Overall Leaderboard (top 15 horizontal bar) ----------
top_n = min(15, len(df_leaderboard))
df_top = df_leaderboard.head(top_n).copy()

fig, ax = plt.subplots(figsize=(10, max(4, top_n * 0.45)))

# create short labels
labels = df_top['experiment_id'].str.replace('EXP_', '', regex=False).tolist()
y_pos = np.arange(top_n)
values = df_top['mean_mae'].values

# colour gradient: best = dark, worst = light
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, top_n))

bars = ax.barh(y_pos, values, color=colors, edgecolor='grey', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Mean MAE (lower is better)')
ax.set_title(f'Top {top_n} Experiments by Mean MAE')

for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=7)

plt.tight_layout()
save_path = os.path.join(FIGURES_DIR, '01_overall_leaderboard.png')
plt.savefig(save_path, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

---
## Section 5: Ablation Analysis

Compare image backbones, text backbones, fusion methods, and loss functions within their respective ablation phases.

In [ ]:
def ablation_comparison(df_all, phase_filter, group_col, title, fig_filename,
                        csv_name=None, include_baseline_id=None):
    """
    Filter experiments by phase, keep the best per group_col,
    plot a bar chart, and save CSV + PNG.

    Parameters
    ----------
    df_all : DataFrame with all experiments
    phase_filter : callable(exp_id) -> bool, or list of phase strings
    group_col : column to group by (e.g. 'image_backbone')
    title : chart title
    fig_filename : filename stem (no extension)
    csv_name : optional CSV filename stem
    include_baseline_id : optional experiment_id to always include
    """
    if callable(phase_filter):
        mask = df_all['experiment_id'].apply(phase_filter)
    else:
        mask = df_all['phase'].isin(phase_filter)

    if include_baseline_id:
        mask = mask | (df_all['experiment_id'] == include_baseline_id)

    df_phase = df_all[mask].copy()
    if df_phase.empty:
        print(f"  [WARN] No experiments found for {title}. Skipping.")
        return pd.DataFrame()

    # keep best per group
    df_best = (df_phase
               .sort_values('mean_mae')
               .drop_duplicates(subset=[group_col], keep='first')
               .sort_values('mean_mae')
               .reset_index(drop=True))

    # save csv
    stem = csv_name or fig_filename
    csv_path = os.path.join(TABLES_DIR, f'{stem}.csv')
    df_best.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path}")

    # plot
    fig, ax = plt.subplots(figsize=(8, max(3, len(df_best) * 0.6)))
    y_pos = np.arange(len(df_best))
    labels = df_best[group_col].tolist()
    values = df_best['mean_mae'].values
    colors = plt.cm.viridis(np.linspace(0.25, 0.85, len(df_best)))

    bars = ax.barh(y_pos, values, color=colors, edgecolor='grey', linewidth=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Mean MAE (lower is better)')
    ax.set_title(title)

    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)

    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, f'{fig_filename}.png')
    plt.savefig(fig_path, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {fig_path}")

    return df_best

In [ ]:
# --- 5a. Image Backbone Comparison (Phase 2 + baseline) ---
print("=== Image Backbone Comparison ===")
df_img = ablation_comparison(
    df_lb,
    phase_filter=lambda eid: 'EXP_020' in eid.upper(),
    group_col='image_backbone',
    title='Image Backbone Comparison (Mean MAE)',
    fig_filename='02_image_backbone_comparison',
    csv_name='image_backbone_comparison',
    include_baseline_id=next(
        (r for r in df_lb['experiment_id'] if 'EXP_012' in r.upper()), None
    ),
)

In [ ]:
# --- 5b. Text Backbone Comparison (Phase 3 + phase-2 winner as XLM-R ref) ---
print("=== Text Backbone Comparison ===")

# include the best EXP_020* as the XLM-R reference
best_020 = df_lb[df_lb['experiment_id'].str.upper().str.contains('EXP_020')]
ref_020_id = best_020.iloc[0]['experiment_id'] if not best_020.empty else None

df_txt = ablation_comparison(
    df_lb,
    phase_filter=lambda eid: 'EXP_030' in eid.upper(),
    group_col='text_backbone',
    title='Text Backbone Comparison (Mean MAE)',
    fig_filename='03_text_backbone_comparison',
    csv_name='text_backbone_comparison',
    include_baseline_id=ref_020_id,
)

In [ ]:
# --- 5c. Fusion Comparison (Phase 4 + concat baseline) ---
print("=== Fusion Method Comparison ===")

# include the best Phase 3 experiment as Concat reference
best_030 = df_lb[df_lb['experiment_id'].str.upper().str.contains('EXP_030')]
ref_030_id = best_030.iloc[0]['experiment_id'] if not best_030.empty else None

df_fus = ablation_comparison(
    df_lb,
    phase_filter=lambda eid: ('EXP_040' in eid.upper() or 'EXP_041' in eid.upper()),
    group_col='fusion_method',
    title='Fusion Method Comparison (Mean MAE)',
    fig_filename='04_fusion_comparison',
    csv_name='fusion_comparison',
    include_baseline_id=ref_030_id,
)

In [ ]:
# --- 5d. Loss Function Comparison (Phase 5 + MSE baseline) ---
print("=== Loss Function Comparison ===")

# include the best Phase 4 experiment as MSE reference
best_041 = df_lb[df_lb['experiment_id'].str.upper().str.contains('EXP_04')]
ref_041_id = best_041.iloc[0]['experiment_id'] if not best_041.empty else None

df_loss = ablation_comparison(
    df_lb,
    phase_filter=lambda eid: ('EXP_050' in eid.upper() or 'EXP_051' in eid.upper()),
    group_col='loss_function',
    title='Loss Function Comparison (Mean MAE)',
    fig_filename='05_loss_comparison',
    csv_name='loss_comparison',
    include_baseline_id=ref_041_id,
)

---
## Section 6: Performance Evolution Across Phases

Show the best `mean_mae` achieved in each experimental phase, ordered chronologically.

In [ ]:
PHASE_ORDER = [
    'Baseline (Text-Only)',
    'Baseline (Image-Only)',
    'Baseline (Multimodal)',
    'Image Ablation',
    'Text Ablation',
    'Fusion Ablation',
    'Loss Ablation',
    'Promising Combination',
    'Seed Validation',
]

evolution_rows = []
for phase_name in PHASE_ORDER:
    df_phase = df_lb[df_lb['phase'] == phase_name]
    if df_phase.empty:
        continue
    best_row = df_phase.sort_values('mean_mae').iloc[0]
    evolution_rows.append({
        'phase': phase_name,
        'best_experiment': best_row['experiment_id'],
        'mean_mae': best_row['mean_mae'],
        'overall_mae': best_row.get('overall_mae', np.nan),
        'r2_overall': best_row.get('r2_overall', np.nan),
    })

df_evo = pd.DataFrame(evolution_rows)

csv_path = os.path.join(TABLES_DIR, 'performance_evolution.csv')
df_evo.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
df_evo

In [ ]:
# ---------- Figure 6: Performance Evolution Line Plot ----------
if len(df_evo) >= 2:
    fig, ax1 = plt.subplots(figsize=(10, 5))

    x = np.arange(len(df_evo))
    short_labels = [p.replace('Baseline ', 'BL ').replace('Promising Combination', 'Promising Comb.')
                    for p in df_evo['phase']]

    color1 = '#2196F3'
    color2 = '#FF9800'

    ax1.plot(x, df_evo['mean_mae'], 'o-', color=color1, linewidth=2, markersize=8, label='Mean MAE')
    for i, val in enumerate(df_evo['mean_mae']):
        ax1.annotate(f'{val:.4f}', (x[i], val), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=7, color=color1)
    ax1.set_ylabel('Mean MAE (lower is better)', color=color1)
    ax1.tick_params(axis='y', labelcolor=color1)

    if 'overall_mae' in df_evo.columns and df_evo['overall_mae'].notna().any():
        ax1.plot(x, df_evo['overall_mae'], 's--', color=color2, linewidth=2, markersize=7,
                 label='Overall MAE')
        for i, val in enumerate(df_evo['overall_mae']):
            if not np.isnan(val):
                ax1.annotate(f'{val:.4f}', (x[i], val), textcoords='offset points',
                             xytext=(0, -14), ha='center', fontsize=7, color=color2)

    ax1.set_xticks(x)
    ax1.set_xticklabels(short_labels, rotation=30, ha='right', fontsize=8)
    ax1.set_title('Performance Evolution Across Phases')
    ax1.legend(loc='upper right', fontsize=8)
    ax1.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(FIGURES_DIR, '06_performance_evolution.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
else:
    print("Not enough phases to plot evolution.")

---
## Section 7: Top-3 Radar Chart

Radar chart of the top 3 experiments across five MAE targets. Scores are normalized so that **higher = better** (score = 1 - normalized MAE).

In [ ]:
radar_aspects = ['mae_overall', 'mae_food', 'mae_price', 'mae_service', 'mae_atmos']
radar_labels = ['Overall', 'Food', 'Price', 'Service', 'Atmosphere']

# ensure all columns exist
available_aspects = [a for a in radar_aspects if a in df_lb.columns]
if len(available_aspects) < 3:
    print("Not enough aspect MAE columns for radar chart.")
else:
    top3 = df_lb.head(min(3, len(df_lb))).copy()

    # normalize: score = 1 - (value - global_min) / (global_max - global_min)
    radar_data = top3[available_aspects].values  # shape (3, n_aspects)
    global_min = df_lb[available_aspects].min().values
    global_max = df_lb[available_aspects].max().values
    denom = global_max - global_min
    denom[denom == 0] = 1.0  # avoid division by zero
    scores = 1.0 - (radar_data - global_min) / denom

    # radar chart
    n_aspects = len(available_aspects)
    angles = np.linspace(0, 2 * np.pi, n_aspects, endpoint=False).tolist()
    angles += angles[:1]  # close polygon

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    radar_colors = ['#2196F3', '#FF5722', '#4CAF50']

    for i in range(len(top3)):
        vals = scores[i].tolist() + [scores[i][0]]
        label = top3.iloc[i]['experiment_id'].replace('EXP_', '')
        ax.plot(angles, vals, 'o-', linewidth=2, label=label, color=radar_colors[i % 3])
        ax.fill(angles, vals, alpha=0.1, color=radar_colors[i % 3])

    used_labels = [radar_labels[radar_aspects.index(a)] for a in available_aspects]
    ax.set_thetagrids(np.degrees(angles[:-1]), used_labels)
    ax.set_ylim(0, 1.05)
    ax.set_title('Top-3 Experiments: Normalized Score\n(higher = better)', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

    plt.tight_layout()
    save_path = os.path.join(FIGURES_DIR, '07_top3_radar_chart.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

---
## Section 8: Validation vs Test Comparison

For experiments that have `test_metrics.json`, compare validation and test performance.

In [ ]:
df_test = df_lb[df_lb['has_test_metrics'] == True].copy()

if df_test.empty:
    print("No experiments with test metrics found. Skipping.")
    df_val_test = pd.DataFrame()
else:
    rows = []
    for _, row in df_test.iterrows():
        r = {
            'experiment_id': row['experiment_id'],
            'phase': row['phase'],
            'val_mean_mae': row.get('mean_mae', np.nan),
            'test_mean_mae': row.get('test_mean_mae', np.nan),
            'val_overall_mae': row.get('overall_mae', np.nan),
            'test_overall_mae': row.get('test_overall_mae', row.get('test_mae_overall', np.nan)),
        }
        r['generalization_gap_mean_mae'] = r['test_mean_mae'] - r['val_mean_mae']
        r['generalization_gap_overall_mae'] = r['test_overall_mae'] - r['val_overall_mae']
        rows.append(r)

    df_val_test = pd.DataFrame(rows)

    csv_path = os.path.join(TABLES_DIR, 'validation_vs_test_comparison.csv')
    xlsx_path = os.path.join(TABLES_DIR, 'validation_vs_test_comparison.xlsx')
    df_val_test.to_csv(csv_path, index=False)
    df_val_test.to_excel(xlsx_path, index=False, engine='openpyxl')
    print(f"Saved: {csv_path}")
    print(f"Saved: {xlsx_path}")
    df_val_test

In [ ]:
# ---------- Validation vs Test bar chart ----------
if len(df_val_test) >= 2:
    fig, ax = plt.subplots(figsize=(10, max(4, len(df_val_test) * 0.8)))

    y_pos = np.arange(len(df_val_test))
    bar_height = 0.35
    labels = df_val_test['experiment_id'].str.replace('EXP_', '', regex=False).tolist()

    bars_val = ax.barh(y_pos - bar_height / 2, df_val_test['val_mean_mae'],
                       bar_height, label='Validation', color='#2196F3', edgecolor='grey', linewidth=0.5)
    bars_test = ax.barh(y_pos + bar_height / 2, df_val_test['test_mean_mae'],
                        bar_height, label='Test', color='#FF5722', edgecolor='grey', linewidth=0.5)

    for bar, val in zip(bars_val, df_val_test['val_mean_mae']):
        if not np.isnan(val):
            ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                    f'{val:.4f}', va='center', fontsize=7)
    for bar, val in zip(bars_test, df_val_test['test_mean_mae']):
        if not np.isnan(val):
            ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                    f'{val:.4f}', va='center', fontsize=7)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Mean MAE')
    ax.set_title('Validation vs Test Performance')
    ax.legend(loc='lower right', fontsize=9)

    plt.tight_layout()
    save_path = os.path.join(FIGURES_DIR, 'validation_vs_test_comparison.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
elif len(df_val_test) == 1:
    print("Only 1 experiment with test metrics. Bar chart requires >= 2. Skipping chart.")
else:
    print("No test metrics available. Skipping chart.")

---
## Section 9: Improvement vs Baseline

Compare every experiment against the multimodal baseline (`EXP_012`).

In [ ]:
# find baseline
baseline_candidates = df_lb[df_lb['experiment_id'].str.upper().str.contains('EXP_012')]
if baseline_candidates.empty:
    # fallback: earliest multimodal concat MSE
    fallback = df_lb[
        (df_lb['fusion_method'] == 'Concat') &
        (df_lb['loss_function'] == 'MSE') &
        (~df_lb['is_unimodal_text']) &
        (~df_lb['is_unimodal_image'])
    ]
    if not fallback.empty:
        baseline_row = fallback.iloc[0]
    else:
        baseline_row = None
else:
    baseline_row = baseline_candidates.iloc[0]

if baseline_row is not None:
    baseline_id = baseline_row['experiment_id']
    baseline_mean_mae = baseline_row['mean_mae']
    print(f"Baseline: {baseline_id}  (mean_mae = {baseline_mean_mae:.4f})")

    df_improve = df_lb[['experiment_id', 'phase', 'mean_mae', 'overall_mae']].copy()
    df_improve['baseline_mean_mae'] = baseline_mean_mae
    df_improve['absolute_improvement'] = baseline_mean_mae - df_improve['mean_mae']
    df_improve['relative_improvement_percent'] = (
        df_improve['absolute_improvement'] / baseline_mean_mae * 100
    )
    df_improve = df_improve.sort_values('absolute_improvement', ascending=False).reset_index(drop=True)

    csv_path = os.path.join(TABLES_DIR, 'improvement_vs_baseline.csv')
    xlsx_path = os.path.join(TABLES_DIR, 'improvement_vs_baseline.xlsx')
    df_improve.to_csv(csv_path, index=False)
    df_improve.to_excel(xlsx_path, index=False, engine='openpyxl')
    print(f"Saved: {csv_path}")
    print(f"Saved: {xlsx_path}")
    df_improve.head(15)
else:
    print("No suitable baseline found. Skipping improvement table.")
    df_improve = pd.DataFrame()

In [ ]:
# ---------- Improvement vs Baseline bar chart ----------
if not df_improve.empty and len(df_improve) >= 2:
    # exclude the baseline itself
    df_plot = df_improve[df_improve['experiment_id'] != baseline_id].copy()
    top_n_imp = min(15, len(df_plot))
    df_plot = df_plot.head(top_n_imp)

    fig, ax = plt.subplots(figsize=(10, max(4, top_n_imp * 0.45)))
    y_pos = np.arange(top_n_imp)
    labels = df_plot['experiment_id'].str.replace('EXP_', '', regex=False).tolist()
    values = df_plot['relative_improvement_percent'].values
    colors = ['#4CAF50' if v >= 0 else '#F44336' for v in values]

    bars = ax.barh(y_pos, values, color=colors, edgecolor='grey', linewidth=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_xlabel('Relative Improvement vs Baseline (%)')
    ax.set_title(f'Improvement over Baseline ({baseline_id})')

    for bar, val in zip(bars, values):
        offset = 0.3 if val >= 0 else -2.5
        ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height() / 2,
                f'{val:+.2f}%', va='center', fontsize=7)

    plt.tight_layout()
    save_path = os.path.join(FIGURES_DIR, 'improvement_vs_baseline.png')
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")
else:
    print("Not enough data for improvement chart.")

---
## Section 5 (cont.): Ablation Summary Table

Summarize each ablation phase: reference experiment, best experiment, absolute & relative improvement.

In [ ]:
def get_best_in_group(df_all, filter_fn):
    """Return the row with the lowest mean_mae matching filter_fn(experiment_id)."""
    sub = df_all[df_all['experiment_id'].apply(filter_fn)]
    if sub.empty:
        return None
    return sub.sort_values('mean_mae').iloc[0]


def first_match(df_all, filter_fn):
    """Return the first row matching filter_fn."""
    sub = df_all[df_all['experiment_id'].apply(filter_fn)]
    if sub.empty:
        return None
    return sub.iloc[0]


ablation_rows = []

# --- Image Ablation ---
ref_img = first_match(df_lb, lambda e: 'EXP_012' in e.upper())
best_img = get_best_in_group(df_lb, lambda e: 'EXP_020' in e.upper())
if ref_img is not None and best_img is not None:
    imp = ref_img['mean_mae'] - best_img['mean_mae']
    ablation_rows.append({
        'phase': 'Image Ablation',
        'reference_experiment': ref_img['experiment_id'],
        'best_experiment': best_img['experiment_id'],
        'reference_mean_mae': ref_img['mean_mae'],
        'best_mean_mae': best_img['mean_mae'],
        'absolute_improvement': imp,
        'relative_improvement_percent': imp / ref_img['mean_mae'] * 100 if ref_img['mean_mae'] != 0 else 0,
        'interpretation': f"Best image backbone: {best_img['image_backbone']}",
    })

# --- Text Ablation ---
ref_txt = best_img  # best from Phase 2 with XLM-R
best_txt = get_best_in_group(df_lb, lambda e: 'EXP_030' in e.upper())
if ref_txt is not None and best_txt is not None:
    imp = ref_txt['mean_mae'] - best_txt['mean_mae']
    ablation_rows.append({
        'phase': 'Text Ablation',
        'reference_experiment': ref_txt['experiment_id'],
        'best_experiment': best_txt['experiment_id'],
        'reference_mean_mae': ref_txt['mean_mae'],
        'best_mean_mae': best_txt['mean_mae'],
        'absolute_improvement': imp,
        'relative_improvement_percent': imp / ref_txt['mean_mae'] * 100 if ref_txt['mean_mae'] != 0 else 0,
        'interpretation': f"Best text backbone: {best_txt['text_backbone']}",
    })

# --- Fusion Ablation ---
ref_fus = best_txt  # best from Phase 3 (Concat)
best_fus = get_best_in_group(df_lb, lambda e: 'EXP_040' in e.upper() or 'EXP_041' in e.upper())
if ref_fus is not None and best_fus is not None:
    imp = ref_fus['mean_mae'] - best_fus['mean_mae']
    ablation_rows.append({
        'phase': 'Fusion Ablation',
        'reference_experiment': ref_fus['experiment_id'],
        'best_experiment': best_fus['experiment_id'],
        'reference_mean_mae': ref_fus['mean_mae'],
        'best_mean_mae': best_fus['mean_mae'],
        'absolute_improvement': imp,
        'relative_improvement_percent': imp / ref_fus['mean_mae'] * 100 if ref_fus['mean_mae'] != 0 else 0,
        'interpretation': f"Best fusion: {best_fus['fusion_method']}",
    })

# --- Loss Ablation ---
ref_loss = best_fus  # best from Phase 4 (MSE)
best_loss_row = get_best_in_group(df_lb, lambda e: 'EXP_050' in e.upper() or 'EXP_051' in e.upper())
if ref_loss is not None and best_loss_row is not None:
    imp = ref_loss['mean_mae'] - best_loss_row['mean_mae']
    ablation_rows.append({
        'phase': 'Loss Ablation',
        'reference_experiment': ref_loss['experiment_id'],
        'best_experiment': best_loss_row['experiment_id'],
        'reference_mean_mae': ref_loss['mean_mae'],
        'best_mean_mae': best_loss_row['mean_mae'],
        'absolute_improvement': imp,
        'relative_improvement_percent': imp / ref_loss['mean_mae'] * 100 if ref_loss['mean_mae'] != 0 else 0,
        'interpretation': f"Best loss: {best_loss_row['loss_function']}",
    })

# --- Promising Combination ---
ref_comb = get_best_in_group(df_lb, lambda e: 'EXP_060A' in e.upper() or 'bestsequential' in e.lower())
if ref_comb is None:
    ref_comb = best_loss_row  # fallback: best sequential
best_comb = get_best_in_group(df_lb, lambda e: 'EXP_060' in e.upper())
if ref_comb is not None and best_comb is not None:
    imp = ref_comb['mean_mae'] - best_comb['mean_mae']
    ablation_rows.append({
        'phase': 'Promising Combination',
        'reference_experiment': ref_comb['experiment_id'],
        'best_experiment': best_comb['experiment_id'],
        'reference_mean_mae': ref_comb['mean_mae'],
        'best_mean_mae': best_comb['mean_mae'],
        'absolute_improvement': imp,
        'relative_improvement_percent': imp / ref_comb['mean_mae'] * 100 if ref_comb['mean_mae'] != 0 else 0,
        'interpretation': 'Best alternative combination',
    })

df_ablation = pd.DataFrame(ablation_rows)

if not df_ablation.empty:
    csv_path = os.path.join(TABLES_DIR, 'ablation_summary.csv')
    xlsx_path = os.path.join(TABLES_DIR, 'ablation_summary.xlsx')
    df_ablation.to_csv(csv_path, index=False)
    df_ablation.to_excel(xlsx_path, index=False, engine='openpyxl')
    print(f"Saved: {csv_path}")
    print(f"Saved: {xlsx_path}")

df_ablation

---
## Section 10: Markdown Report

Generate `experiment_report.md` summarizing key findings.

In [ ]:
def fmt(val, decimals=4):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return 'N/A'
    return f'{val:.{decimals}f}'


lines = []
lines.append('# Experiment Report')
lines.append('')
lines.append(f'**Generated automatically** | Total experiments: {len(df_lb)}')
lines.append('')

# --- Best model ---
best = df_lb.iloc[0]
lines.append('## Best Model')
lines.append('')
lines.append(f'| Property | Value |')
lines.append(f'|----------|-------|')
lines.append(f'| Experiment | `{best["experiment_id"]}` |')
lines.append(f'| Phase | {best["phase"]} |')
lines.append(f'| Image Backbone | {best["image_backbone"]} |')
lines.append(f'| Text Backbone | {best["text_backbone"]} |')
lines.append(f'| Fusion | {best["fusion_method"]} |')
lines.append(f'| Loss | {best["loss_function"]} |')
lines.append(f'| Mean MAE | {fmt(best.get("mean_mae"))} |')
lines.append(f'| Overall MAE | {fmt(best.get("overall_mae"))} |')
lines.append(f'| R\u00b2 Overall | {fmt(best.get("r2_overall"))} |')
lines.append('')

# --- Phase-by-phase results ---
lines.append('## Best Result per Phase')
lines.append('')
lines.append('| Phase | Best Experiment | Mean MAE | Overall MAE |')
lines.append('|-------|-----------------|----------|-------------|')
for _, row in df_evo.iterrows():
    lines.append(f'| {row["phase"]} | `{row["best_experiment"]}` | {fmt(row["mean_mae"])} | {fmt(row.get("overall_mae"))} |')
lines.append('')

# --- Ablation summary ---
if not df_ablation.empty:
    lines.append('## Ablation Summary')
    lines.append('')
    lines.append('| Phase | Reference | Best | Ref MAE | Best MAE | Improvement |')
    lines.append('|-------|-----------|------|---------|----------|-------------|')
    for _, row in df_ablation.iterrows():
        lines.append(
            f'| {row["phase"]} | `{row["reference_experiment"]}` | '
            f'`{row["best_experiment"]}` | {fmt(row["reference_mean_mae"])} | '
            f'{fmt(row["best_mean_mae"])} | {fmt(row["relative_improvement_percent"], 2)}% |'
        )
    lines.append('')

# --- Baseline improvement ---
if not df_improve.empty and baseline_row is not None:
    lines.append('## Improvement vs Baseline')
    lines.append('')
    lines.append(f'Baseline: `{baseline_id}` (mean_mae = {fmt(baseline_mean_mae)})')
    lines.append('')
    top5_imp = df_improve.head(5)
    lines.append('| Experiment | Mean MAE | Improvement |')
    lines.append('|------------|----------|-------------|')
    for _, row in top5_imp.iterrows():
        lines.append(
            f'| `{row["experiment_id"]}` | {fmt(row["mean_mae"])} | '
            f'{fmt(row["relative_improvement_percent"], 2)}% |'
        )
    lines.append('')

# --- Test comparison ---
if not df_val_test.empty:
    lines.append('## Validation vs Test')
    lines.append('')
    lines.append('| Experiment | Val Mean MAE | Test Mean MAE | Gap |')
    lines.append('|------------|-------------|--------------|-----|')
    for _, row in df_val_test.iterrows():
        lines.append(
            f'| `{row["experiment_id"]}` | {fmt(row["val_mean_mae"])} | '
            f'{fmt(row["test_mean_mae"])} | {fmt(row["generalization_gap_mean_mae"])} |'
        )
    lines.append('')

report_text = '\n'.join(lines)

report_path = os.path.join(REPORTS_DIR, 'experiment_report.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

print(f"Saved: {report_path}")
print()
print(report_text[:2000])

---
## Section 11: Logic Checks

Validate data integrity and confirm all outputs were saved.

In [ ]:
checks = []

# 1. number of discovered experiments
checks.append(('Discovered folders', len(folders)))
checks.append(('Loaded metrics', len(experiment_records)))

# 2. missing metrics
checks.append(('Skipped (no metrics.json)', len(skipped_folders)))
if skipped_folders:
    for s in skipped_folders:
        checks.append((f'  - {s}', 'MISSING'))

# 3. duplicated IDs
dup_count = df_lb['experiment_id'].duplicated().sum()
checks.append(('Duplicated experiment IDs', dup_count))

# 4. required metric columns
required_cols = ['mean_mae', 'overall_mae', 'mae_food', 'mae_price', 'mae_atmos', 'mae_service']
for col in required_cols:
    n_missing = df_lb[col].isna().sum() if col in df_lb.columns else len(df_lb)
    status = 'OK' if n_missing == 0 else f'{n_missing} missing'
    checks.append((f'Column: {col}', status))

# 5. baseline exists
checks.append(('Baseline found', 'YES' if baseline_row is not None else 'NO'))

# 6. Phase 6 test metrics
n_test = df_lb['has_test_metrics'].sum()
checks.append(('Experiments with test metrics', int(n_test)))

# 7. output files
expected_files = [
    os.path.join(FIGURES_DIR, '01_overall_leaderboard.png'),
    os.path.join(FIGURES_DIR, '02_image_backbone_comparison.png'),
    os.path.join(FIGURES_DIR, '03_text_backbone_comparison.png'),
    os.path.join(FIGURES_DIR, '04_fusion_comparison.png'),
    os.path.join(FIGURES_DIR, '05_loss_comparison.png'),
    os.path.join(FIGURES_DIR, '06_performance_evolution.png'),
    os.path.join(FIGURES_DIR, '07_top3_radar_chart.png'),
    os.path.join(TABLES_DIR, 'full_experiment_leaderboard.csv'),
    os.path.join(TABLES_DIR, 'full_experiment_leaderboard.xlsx'),
    os.path.join(TABLES_DIR, 'ablation_summary.csv'),
    os.path.join(TABLES_DIR, 'ablation_summary.xlsx'),
    os.path.join(TABLES_DIR, 'improvement_vs_baseline.csv'),
    os.path.join(TABLES_DIR, 'improvement_vs_baseline.xlsx'),
    os.path.join(REPORTS_DIR, 'experiment_report.md'),
]
# conditional files
if not df_val_test.empty:
    expected_files += [
        os.path.join(TABLES_DIR, 'validation_vs_test_comparison.csv'),
        os.path.join(TABLES_DIR, 'validation_vs_test_comparison.xlsx'),
    ]
    if len(df_val_test) >= 2:
        expected_files.append(os.path.join(FIGURES_DIR, 'validation_vs_test_comparison.png'))

for fp in expected_files:
    exists = os.path.isfile(fp)
    checks.append((os.path.basename(fp), 'SAVED' if exists else 'MISSING'))

print('=== Logic Checks ===')
for label, value in checks:
    print(f'  {label:.<55s} {value}')

# summary
n_missing_files = sum(1 for _, v in checks if v == 'MISSING')
print(f'\nTotal missing output files: {n_missing_files}')
if n_missing_files == 0 and dup_count == 0:
    print('All checks passed.')
else:
    print('Some checks need attention — review warnings above.')